In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

print("Notebook 02: Preprocessing started")

# ──── 1. Load raw data ────────────────────────────────────────────
RAW_PATH = '../data/raw/application_train.csv'
print("Loading:", RAW_PATH)

df = pd.read_csv(RAW_PATH)
print("Original shape:", df.shape)

Notebook 02: Preprocessing started
Loading: ../data/raw/application_train.csv
Original shape: (307511, 122)


In [3]:
print(df.columns)

Index(['SK_ID_CURR', 'TARGET', 'NAME_CONTRACT_TYPE', 'CODE_GENDER',
       'FLAG_OWN_CAR', 'FLAG_OWN_REALTY', 'CNT_CHILDREN', 'AMT_INCOME_TOTAL',
       'AMT_CREDIT', 'AMT_ANNUITY',
       ...
       'FLAG_DOCUMENT_18', 'FLAG_DOCUMENT_19', 'FLAG_DOCUMENT_20',
       'FLAG_DOCUMENT_21', 'AMT_REQ_CREDIT_BUREAU_HOUR',
       'AMT_REQ_CREDIT_BUREAU_DAY', 'AMT_REQ_CREDIT_BUREAU_WEEK',
       'AMT_REQ_CREDIT_BUREAU_MON', 'AMT_REQ_CREDIT_BUREAU_QRT',
       'AMT_REQ_CREDIT_BUREAU_YEAR'],
      dtype='str', length=122)


In [ ]:
# ──── 2. Stratified sampling (very important for speed) ───────────
SAMPLE_FRAC = 0.15

print(f"Sampling {SAMPLE_FRAC*100:.1f}% of data (stratified on TARGET)...")

df_sampled, _ = train_test_split(
    df,
    train_size=SAMPLE_FRAC,
    random_state=42,
    stratify=df['TARGET']
)

print("After sampling shape:", df_sampled.shape)

Sampling 15.0% of data (stratified on TARGET)...
After sampling shape: (46126, 122)


In [5]:
# ──── 3. Basic cleaning & anomaly handling ────────────────────────
print("Cleaning anomalies...")
df['DAYS_EMPLOYED'] = df['DAYS_EMPLOYED'].replace(365243, np.nan)

Cleaning anomalies...


In [6]:
# ──── 4. Simple feature engineering ───────────────────────────────
print("Adding basic engineered features...")
df['CREDIT_INCOME_RATIO']     = df['AMT_CREDIT'] / df['AMT_INCOME_TOTAL']
df['ANNUITY_INCOME_RATIO']    = df['AMT_ANNUITY'] / df['AMT_INCOME_TOTAL']
df['EMPLOYED_AGE_RATIO']      = df['DAYS_EMPLOYED'] / df['DAYS_BIRTH']
df['CREDIT_DURATION_RATIO']   = df['AMT_CREDIT'] / df['AMT_ANNUITY'].replace(0, np.nan)

Adding basic engineered features...


C:\Users\sharo\AppData\Local\Temp\ipykernel_34384\3552739021.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['CREDIT_INCOME_RATIO']     = df['AMT_CREDIT'] / df['AMT_INCOME_TOTAL']
C:\Users\sharo\AppData\Local\Temp\ipykernel_34384\3552739021.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['ANNUITY_INCOME_RATIO']    = df['AMT_ANNUITY'] / df['AMT_INCOME_TOTAL']
C:\Users\sharo\AppData\Local\Temp\ipykernel_34384\3552739021.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `

In [7]:
# ──── 5. Drop unnecessary / very high-missing columns ─────────────
print("Dropping ID and columns with >50% missing...")
drop_cols = ['SK_ID_CURR'] + df.columns[df.isnull().mean() > 0.50].tolist()
df = df.drop(columns=drop_cols)
print("Shape after dropping:", df.shape)

Dropping ID and columns with >50% missing...
Shape after dropping: (307511, 84)


In [8]:
# ──── 6. Separate features and target ─────────────────────────────
X = df.drop('TARGET', axis=1)
y = df['TARGET']

print("Features shape:", X.shape)
print("Target distribution:\n", y.value_counts(normalize=True))

# ──── 7. Identify numeric and categorical columns ─────────────────
num_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_cols = X.select_dtypes(include=['object']).columns.tolist()

print(f"Numeric columns: {len(num_cols)}")
print(f"Categorical columns: {len(cat_cols)}")

Features shape: (307511, 83)
Target distribution:
 TARGET
0    0.919271
1    0.080729
Name: proportion, dtype: float64
Numeric columns: 70
Categorical columns: 13


C:\Users\sharo\AppData\Local\Temp\ipykernel_34384\2180604842.py:10: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X.select_dtypes(include=['object']).columns.tolist()


In [14]:
# ──── 8. Build preprocessing pipeline ─────────────────────────────
num_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler())
])

cat_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('onehot',  OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', num_transformer, num_cols),
        ('cat', cat_transformer, cat_cols)
    ],
    remainder='drop'
)

In [10]:
# ──── 9. Train/test split (stratified) ─────────────────────────────
print("Splitting data...")
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

print("X_train shape:", X_train.shape)
print("X_test shape :", X_test.shape)

Splitting data...
X_train shape: (246008, 83)
X_test shape : (61503, 83)


In [11]:
# ──── 10. Fit & transform ─────────────────────────────────────────
print("Applying preprocessing...")
X_train_proc = preprocessor.fit_transform(X_train)
X_test_proc  = preprocessor.transform(X_test)

print("Processed shapes:")
print("  X_train_proc:", X_train_proc.shape)
print("  X_test_proc :", X_test_proc.shape)

Applying preprocessing...
Processed shapes:
  X_train_proc: (246008, 199)
  X_test_proc : (61503, 199)


In [12]:
# ──── 11. Convert to DataFrame & save ─────────────────────────────
# Use your current naming convention
X_train_df = pd.DataFrame(X_train_proc)
X_test_df  = pd.DataFrame(X_test_proc)

X_train_df.to_csv('../data/processed/X_train.csv', index=False)
X_test_df.to_csv('../data/processed/X_test.csv', index=False)

y_train.to_csv('../data/processed/y_train.csv', index=False)
y_test.to_csv('../data/processed/y_test.csv', index=False)

print("\nFiles saved:")
print("  ../data/processed/X_train.csv")
print("  ../data/processed/X_test.csv")
print("  ../data/processed/y_train.csv")
print("  ../data/processed/y_test.csv")

print("\nPreprocessing complete. Ready for model training.")


Files saved:
  ../data/processed/X_train.csv
  ../data/processed/X_test.csv
  ../data/processed/y_train.csv
  ../data/processed/y_test.csv

Preprocessing complete. Ready for model training.


In [13]:
# Save the preprocessor so we can use it later for new inputs
import joblib
joblib.dump(preprocessor, '../models/preprocessor.pkl')
print("Preprocessor saved to ../models/preprocessor.pkl")

Preprocessor saved to ../models/preprocessor.pkl
